# 01 -- Breakdown radius examples

Walk through `delta_opt` and `delta_valid` on small graphs where the answer can
be checked by hand, before trusting either on anything larger.


In [ ]:
import numpy as np

from bkrobust.graphs.mpdag import MPDAG
from bkrobust.graphs import adjustment
from bkrobust.theory import radius
from bkrobust.theory.certificate import certify
from bkrobust.data import synthetic

## A graph where O* is not the back-door set

`Z1 -> X`, `Z1 -> Y`, `Z2 -> Y`, `X -> M -> Y`, `X -> Y`.

The canonical back-door set is `{Z1}`. `O*` is `{Z1, Z2}` -- `Z2` does not touch
the treatment, so it changes nothing about validity, but it explains outcome
variance and shrinks the residual variance of the adjusted estimator. Optimality
is a variance claim, not an identification claim.


In [ ]:
g = MPDAG(
    nodes=["Z1", "Z2", "X", "M", "Y"],
    directed=[("Z1", "X"), ("Z1", "Y"), ("Z2", "Y"), ("X", "M"), ("M", "Y"), ("X", "Y")],
)
cpdag = synthetic.dag_to_cpdag(g)

adjustment.canonical_adjustment_set(g, "X", "Y"), adjustment.optimal_adjustment_set(g, "X", "Y")

## Both radii

Exact enumeration -- small enough. `delta_opt` needs an SCM, since optimality is
a statement about variance and not about the graph alone.


In [ ]:
rng = np.random.default_rng(0)
scm = synthetic.random_scm(g, rng)

valid, opt = radius.radius_report(cpdag, g, "X", "Y", scm=scm, method="exact")
valid, opt

## The ordering, and the witnesses

`verify_ordering` checks `delta_opt <= delta_valid`. A violation on an **exact**
pair of reports is a counterexample to theorem target T1 -- keep the witness.


In [ ]:
radius.verify_ordering(valid, opt)

In [ ]:
valid.witness, opt.witness

## Worst case versus typical case

The radii say the ball at `delta` *contains* a breaking knowledge set.
`radius_distribution` says how much of it breaks. A radius of 2 where 1% of the
ball breaks is a very different practical situation from one where 80% does.


In [ ]:
radius.radius_distribution(cpdag, g, "X", "Y", delta=1, scm=scm)

## The certificate

What a practitioner can compute without the true DAG. Note the `exact` flag and
the load-bearing constraint list.


In [ ]:
from bkrobust.knowledge.base import BackgroundKnowledge

bk = BackgroundKnowledge().with_edge("Z1", "X")
print(certify(cpdag, bk, "X", "Y", scm=scm))